# Spark 4.1 + Kafka connectivity check

This notebook assumes you started services on the same docker compose network:

```bash
docker compose -f docker-compose-spark41.yml -f docker-compose-kafka.yml -f docker-compose-notebooks.yml up -d --build
```

- Spark master: `spark://spark-master-41:7078`
- Kafka: `kafka:29092`


In [5]:
import os
from pyspark.sql import SparkSession

spark_master = os.environ.get("SPARK_MASTER", "spark://spark-master-41:7078")
kafka_bootstrap = os.environ.get("KAFKA_BOOTSTRAP_SERVERS", "kafka:29092")
topic = os.environ.get("KAFKA_TOPIC", "events")

spark = (
    SparkSession.builder
    .appName("spark41-kafka-connectivity")
    .master(spark_master)
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0")
    .getOrCreate()
)

print("Spark version:", spark.version)
print("Spark master:", spark_master)
print("Kafka bootstrap:", kafka_bootstrap)
print("Topic:", topic)


Spark version: 4.1.0
Spark master: spark://spark-master-41:7078
Kafka bootstrap: kafka:29092
Topic: events


In [2]:
# Batch read check (fails fast if topic/broker is wrong)
df = (
    spark.read.format("kafka")
    .option("kafka.bootstrap.servers", kafka_bootstrap)
    .option("subscribe", topic)
    .option("startingOffsets", "earliest")
    .load()
)

df.selectExpr("CAST(key AS STRING) AS key", "CAST(value AS STRING) AS value").show(10, truncate=False)


[Stage 0:>                                                          (0 + 1) / 1]

+----+-----+
|key |value|
+----+-----+
|NULL|hello|
+----+-----+



In [4]:
spark

In [6]:
spark.sparkContext.master


'spark://spark-master-41:7078'